# Threshold Tuning for Face Verification (Cosine Similarity)

**Goal**: Experiment with cosine similarity thresholds to determine the optimal verification threshold for ArcFace or FaceNet embeddings.

**What you will do**
- Load embeddings and identity labels (ArcFace/FaceNet)
- Compute genuine vs imposter similarity scores
- Evaluate FAR, FRR, and Accuracy across thresholds
- Plot metrics and estimate the Equal Error Rate (EER)
- Recommend a threshold with academic justification

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load Embeddings and Labels (ArcFace/FaceNet)

Load precomputed embeddings and identity labels. Ensure embeddings come from ArcFace or FaceNet. This notebook supports:
- `.npy` embeddings and `.csv` labels
- or a simulated dataset if files are not available

In [ ]:
from pathlib import Path

# Update these paths if you already have saved embeddings and labels.
EMBEDDINGS_PATH = Path("data/embeddings_arcface.npy")
LABELS_PATH = Path("data/labels.csv")  # CSV with a column named "identity"


def load_or_simulate_embeddings():
    if EMBEDDINGS_PATH.exists() and LABELS_PATH.exists():
        embeddings = np.load(EMBEDDINGS_PATH)
        labels_df = pd.read_csv(LABELS_PATH)
        labels = labels_df["identity"].astype(str).to_numpy()
        source = "loaded"
    else:
        # Simulated example: 40 identities, 6 samples each, 512-d embeddings
        num_identities = 40
        samples_per_id = 6
        dim = 512
        base = np.random.randn(num_identities, dim)
        base = base / np.linalg.norm(base, axis=1, keepdims=True)
        embeddings = []
        labels = []
        for i in range(num_identities):
            for _ in range(samples_per_id):
                vec = base[i] + 0.15 * np.random.randn(dim)
                vec = vec / np.linalg.norm(vec)
                embeddings.append(vec)
                labels.append(f"id_{i:02d}")
        embeddings = np.vstack(embeddings)
        labels = np.array(labels)
        source = "simulated"
    return embeddings, labels, source


embeddings, labels, source = load_or_simulate_embeddings()
print(f"Embeddings source: {source}")
print("Embeddings shape:", embeddings.shape)
print("Unique identities:", len(np.unique(labels)))

## 2. Compute Cosine Similarities for Genuine vs Imposter Pairs

Create genuine pairs (same identity) and imposter pairs (different identity). Compute cosine similarity for each pair using:

$$
\cos(\theta)=\frac{a\cdot b}{\|a\|\|b\|}
$$

In [ ]:
from itertools import combinations


def cosine_similarity(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def build_pairs(embeddings, labels, max_pairs_per_identity=30, max_imposters=2000):
    # Index by identity
    label_to_indices = {}
    for idx, lab in enumerate(labels):
        label_to_indices.setdefault(lab, []).append(idx)

    genuine_pairs = []
    for lab, idxs in label_to_indices.items():
        if len(idxs) < 2:
            continue
        combs = list(combinations(idxs, 2))
        np.random.shuffle(combs)
        combs = combs[:max_pairs_per_identity]
        genuine_pairs.extend(combs)

    # Imposter pairs: sample across different identities
    all_indices = list(range(len(labels)))
    imposter_pairs = set()
    attempts = 0
    while len(imposter_pairs) < max_imposters and attempts < max_imposters * 10:
        i, j = np.random.choice(all_indices, size=2, replace=False)
        if labels[i] != labels[j]:
            pair = (min(i, j), max(i, j))
            imposter_pairs.add(pair)
        attempts += 1

    return genuine_pairs, list(imposter_pairs)


def pairs_to_scores(embeddings, pairs):
    scores = []
    for i, j in pairs:
        scores.append(cosine_similarity(embeddings[i], embeddings[j]))
    return np.array(scores)


genuine_pairs, imposter_pairs = build_pairs(embeddings, labels)

genuine_scores = pairs_to_scores(embeddings, genuine_pairs)
imposter_scores = pairs_to_scores(embeddings, imposter_pairs)

print("Genuine pairs:", len(genuine_scores))
print("Imposter pairs:", len(imposter_scores))
print("Score ranges:")
print("  Genuine:  min", genuine_scores.min(), "max", genuine_scores.max())
print("  Imposter: min", imposter_scores.min(), "max", imposter_scores.max())

## 3. Evaluate Threshold Grid (0.3-0.7) and Metrics (FAR/FRR/Accuracy)

For thresholds $[0.3, 0.4, 0.5, 0.6, 0.7]$, compute FAR, FRR, and Accuracy:

$$
\text{FAR}=\frac{FP}{FP+TN},\quad
\text{FRR}=\frac{FN}{FN+TP},\quad
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}
$$

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]


def compute_metrics(genuine_scores, imposter_scores, threshold):
    # Accept if score >= threshold
    tp = np.sum(genuine_scores >= threshold)
    fn = np.sum(genuine_scores < threshold)
    fp = np.sum(imposter_scores >= threshold)
    tn = np.sum(imposter_scores < threshold)

    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0

    return tp, fn, fp, tn, far, frr, acc


rows = []
for t in thresholds:
    tp, fn, fp, tn, far, frr, acc = compute_metrics(genuine_scores, imposter_scores, t)
    rows.append({
        "Threshold": t,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
        "FAR": far,
        "FRR": frr,
        "Accuracy": acc,
    })

metrics_df = pd.DataFrame(rows)
metrics_df

## 4. Plot FAR, FRR, and Accuracy vs Threshold

Plot line charts of FAR, FRR, and Accuracy against the threshold values.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(metrics_df["Threshold"], metrics_df["FAR"], marker="o", label="FAR")
plt.plot(metrics_df["Threshold"], metrics_df["FRR"], marker="o", label="FRR")
plt.plot(metrics_df["Threshold"], metrics_df["Accuracy"], marker="o", label="Accuracy")

plt.title("Threshold Tuning: FAR, FRR, Accuracy")
plt.xlabel("Threshold")
plt.ylabel("Rate")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Approximate EER from FAR-FRR Intersection

Find the threshold where $|\text{FAR}-\text{FRR}|$ is minimized and report the corresponding EER value.

In [ ]:
metrics_df["abs_diff"] = (metrics_df["FAR"] - metrics_df["FRR"]).abs()

best_eer_row = metrics_df.loc[metrics_df["abs_diff"].idxmin()]

approx_eer_threshold = float(best_eer_row["Threshold"])
approx_eer = float((best_eer_row["FAR"] + best_eer_row["FRR"]) / 2)

print("Approximate EER threshold:", approx_eer_threshold)
print("Approximate EER value:", approx_eer)

## 6. Threshold Recommendation (Evidence-Based)

**Why low thresholds increase FAR**: When the threshold is low, more pairs are accepted as matches. This includes more imposter pairs whose similarities lie above the low cutoff, increasing false accepts (FAR).

**Why high thresholds increase FRR**: When the threshold is high, even genuine pairs with slightly lower similarity scores are rejected, increasing false rejects (FRR).

**Recommendation**: Use the threshold that balances FAR and FRR (near EER) while maintaining high accuracy. This is often a good operational compromise for biometric verification systems.

In [ ]:
# Suggest a threshold based on max accuracy and EER proximity
best_acc_row = metrics_df.loc[metrics_df["Accuracy"].idxmax()]

print("Best accuracy threshold:", float(best_acc_row["Threshold"]))
print("Best accuracy:", float(best_acc_row["Accuracy"]))
print("EER threshold (approx):", approx_eer_threshold)

### Conclusion

Based on the computed metrics, select a threshold that aligns with your project priorities:
- If minimizing security risk, favor a slightly higher threshold (lower FAR).
- If minimizing user friction, favor a slightly lower threshold (lower FRR).
- For balanced performance, choose the threshold near the EER or the peak accuracy if it does not overly increase FAR.

State your final choice and provide the metric values from the table above for your report.